In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

In [ ]:
import pandas as pd

from qlinks.caging import (
    AdaptiveRegionProposal,
    LocalQDMCageSearchConfig,
    StripeRegionProposal,
    LocalQDMMultiPaddingConfig,
    find_multi_qdm_block_paddings,
    cage_state_to_full_vector,
    certify_qdm_multi_block_result,
    run_local_region_proposal,
    robust_qdm_local_cage_search,
    RobustQDMLocalCageSearchConfig,
    diagnose_qdm_multi_block_paddings,
)
from qlinks.caging.classification import (
    classify_cage_state,
    CageClassificationConfig,
)
from qlinks.models import (
    SquareQLMModel,
    SquareQDMModel,
    TriangularQLMModel,
    TriangularQDMModel,
    HoneycombQLMModel,
    HoneycombQDMModel,
)
from qlinks.visualizer import (
    BasisGridVisualizer,
    HamiltonianGraphVisualizer,
    HamiltonianGraphStyle,
    LinkVisualStyle
)

## Model definition

In [ ]:
model = TriangularQDMModel(
    lx=4,
    ly=4,
    boundary_condition="periodic",
    winding_a=0,
    winding_b=0,
    coup_kin=-1.0,
    coup_pot=1.0,
)

## Find caged states candidate from the seed plaquettes with halo

In [ ]:
local_config = LocalQDMCageSearchConfig(
    halo_layers=0,
    boundary_mode="relaxed",
    tolerance=1.0e-10,
    prune_inactive_local_basis_states=True,
    max_local_states=None,  # set this to cap expensive regions
    degenerate_basis_strategy="ipr",
)

search_config = RobustQDMLocalCageSearchConfig(
    local_config=local_config,
    region_strategies=("snake_stripe",),
    # stripe_widths=(1,),
    # stripe_directions=(0, ),
    # adaptive_beam_width=16,
    # adaptive_branch_factor=64,
    # adaptive_seed_plaquette_ids=(0, 4, 6),
    max_region_plaquettes=8,
    min_region_plaquettes=2,
    max_regions_per_strategy=128,
    snake_stripe_max_turns=None,
    snake_stripe_allow_kind_changes=False,
    snake_stripe_plaquette_kinds=None,
    block_signatures=((0, 4),),
    max_records_per_region=2,
    min_blocks=1,
    max_blocks=3,
    max_product_support_size=16,
    max_paddings_per_stage=32,
    max_padding_attempts_per_stage=32,
    max_paddings_per_packing=16,
    include_sectors=True,
    padding_stages=("static",),
    tolerance=1.0e-9,
    store_full_states=False,
)

## Certify the candidates can be embedded into the full lattice

In [ ]:
certified, context = robust_qdm_local_cage_search(
    model,
    config=search_config,
    return_context=True,
)

print("number of cages:", len(certified))
print("signatures:", certified.signatures)
print("counts by signature:", certified.counts_by_signature)

In [ ]:
print(
    context.n_padding_attempts_by_stage, "\n",
    context.first_certified_attempt_by_stage, "\n",
    context.n_certified_by_stage, "\n",
    context.failure_counts_by_stage, "\n",
    context.leakage_failure_counts_by_stage, "\n",
    context.leakage_failure_norms_by_stage, "\n",
    certified.counts_by_signature, "\n",
)

In [ ]:
signature = (0, 8)
record_index = 0
record = certified[signature, record_index]

print("signature:", record.signature)
print("kappa:", record.kappa)
print("potential value:", record.potential_value)
print("support size:", record.cage_state.support_size)
print("energy:", record.cage_state.energy)
print("boundary residual:", record.cage_state.boundary_residual)
print("eigen residual:", record.cage_state.eigen_residual)
print("full residual:", record.cage_state.full_residual)

## Classify the caged state

Important caveat: the classification report is computed on the limited certified Hilbert space. It can not reliably tell whether the reduced-IZ probe is projective or q_empty

In [ ]:
classification_report = classify_cage_state(
    record.cage_state,
    kinetic_matrix=certified.kinetic_matrix,
    basis_configs=certified.basis.states,
    hilbert_size=certified.hilbert_size,
    config=CageClassificationConfig(
        amplitude_tolerance=1e-10,
        cancellation_tolerance=1e-9,
        action_tolerance=1e-9,
        sector_policy="infer_support_component",
    ),
)

print(classification_report)
print("n zeros:", classification_report.n_nontrivial_zeros)

## Visualize the basis states

In [ ]:
grid_visualizer = BasisGridVisualizer(
    lattice=model.lattice,
    layout=model.layout,
    periodic_image_mode="positive_patch",
    collapse_duplicate_visual_links=True,
    # coordinate_transform=np.array(
    #     [
    #         [1.0, 0.0],
    #         [0.0, 0.72],
    #     ],
    #     dtype=float,
    # ),
    style=LinkVisualStyle(
        node_size=50.0,
        site_label_fontsize=4,
        link_label_fontsize=8,
        plaquette_symbol_fontsize=8,
        vulnerable_link_arrow_length_fraction=1.1,
    ),
)

In [ ]:
grid_visualizer.plot_cage_support(
    record,
    basis_configs=certified.basis.states,
    ncols=4,
    show_amplitudes=True,
    amplitude_digits=3,
    show_config_label=False,
    max_states=None,
)

In [ ]:
grid_visualizer.plot_interference_zeros(
    classification_report,
    ncols=4,
    basis_configs=certified.basis.states,
    mechanism="all",
    max_states=None,
)

## Visualize the subgraph

In [ ]:
if record.full_state is not None:
    psi = record.full_state
else:
    psi = cage_state_to_full_vector(record.cage_state, certified.hilbert_size)

print("psi shape:", psi.shape)
print("limited Hilbert size:", certified.hilbert_size)

graph_viz = HamiltonianGraphVisualizer.cage_subgraph_from_sparse_matrix(
    certified.kinetic_matrix,
    state_vector=psi,
    classification_report=classification_report,
    support_tolerance=1e-10,
    include_zero_edges=True,
    include_self_loops=False,
    weight_tolerance=1e-12,
    style=HamiltonianGraphStyle(
        cmap="coolwarm",
        label_vertices=True,
        vertex_size=20,
    ),
)

graph_viz.plot(
    backend="igraph-mpl",
    layout="kk",
    color_by="state_amplitude_real",
    state_vector=graph_viz.graph_data.state_vector,
    title=f"Cage subgraph, signature={record.signature}",
)